# Split Sessions Pipeline
This notebook reads the massive raw `meet_rtp_records.csv` and `teams_rtp_records.csv` packet dumps and perfectly splits them into individual isolated sessions.

It uses the exact same `client_ip` extraction and 120-second timeout logic as the Rust capture engine.

In [ ]:
import os
import csv

### 1. Define the Server IPs
These are the subnets used by Microsoft Teams and Google Meet, copied directly from `network_metrics.rs`.

In [ ]:
MICROSOFT_SERVER_IPS = [
    "52.112.", "52.113.", "52.114.", "52.115.", "52.122.", "52.123.",
    "2603:1010", "2603:1027", "2603:1037", "2603:1047", "2603:1057", "2620:1ec"
]

MEET_SERVER_IPS = [
    "74.125.250.", "74.125.247.128", "142.250.82.",
    "2001:4860:4864:5:", "2001:4860:4864:4:8000:", "2001:4860:4864:6:"
]

### 2. Client IP Extraction Logic
The function below checks if `src_ip` belongs to a server. If it does, `dst_ip` is our unique client. Otherwise, `src_ip` is the client.

In [ ]:
def is_teams_server(ip):
    return any(ip.startswith(prefix) for prefix in MICROSOFT_SERVER_IPS)

def is_meet_server(ip):
    return any(ip.startswith(prefix) for prefix in MEET_SERVER_IPS)

def get_client_ip(src_ip, dst_ip, platform):
    if platform == "Teams":
        if is_teams_server(src_ip):
            return dst_ip
        return src_ip
    else:
        if is_meet_server(src_ip):
            return dst_ip
        return src_ip

### 3. The Core Splitting Function
This reads the CSV packet by packet (to save RAM) and routes each packet to the correct `client_ip` CSV file.

In [ ]:
# 120-second timeout (matches Rust implementation)
SESSION_TIMEOUT_NS = 120 * 1_000_000_000

def split_csv(input_file, platform, output_dir):
    if not os.path.exists(input_file):
        print(f"[WARN] File {input_file} not found. Skipping...")
        return

    os.makedirs(output_dir, exist_ok=True)
    active_sessions = {}
    
    with open(input_file, 'r') as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            return # empty file
        
        count = 0
        for row in reader:
            if not row or len(row) < 3: continue
            
            arrival_epoch_ns = int(row[0])
            src_ip = row[1]
            dst_ip = row[2]
            
            # 1. Identify the Client IP
            client_ip = get_client_ip(src_ip, dst_ip, platform)
            
            # 2. Check for 120-second silence gap (Timeout)
            if client_ip in active_sessions:
                last_seen = active_sessions[client_ip]['last_seen']
                if arrival_epoch_ns - last_seen > SESSION_TIMEOUT_NS:
                    active_sessions[client_ip]['file_handle'].close()
                    del active_sessions[client_ip]
            
            # 3. Create new session file if it doesn't exist
            if client_ip not in active_sessions:
                # Replace colons to avoid issues with IPv6 filenames
                safe_ip = client_ip.replace(':', '_')
                filename = f"Session_{safe_ip}_{arrival_epoch_ns}.csv"
                filepath = os.path.join(output_dir, filename)
                
                file_handle = open(filepath, 'w', newline='')
                writer = csv.writer(file_handle)
                writer.writerow(header) # Write CSV Header
                
                active_sessions[client_ip] = {
                    'last_seen': arrival_epoch_ns,
                    'file_handle': file_handle,
                    'writer': writer
                }
            
            # 4. Write the packet to the correct session
            active_sessions[client_ip]['writer'].writerow(row)
            active_sessions[client_ip]['last_seen'] = arrival_epoch_ns
            
            count += 1
            if count % 100000 == 0:
                print(f"  ... processed {count} packets for {platform}")

    # Close all remaining open files
    for state in active_sessions.values():
        state['file_handle'].close()
        
    print(f"[SUCCESS] Finished splitting {platform}. Total packets: {count}\n")

### 4. Execute the Pipeline
Make sure your raw CSV files are in the same directory as this notebook, then run this cell!

In [ ]:
# Create the major folder
base_dir = "Sessions_18_to_19"
os.makedirs(base_dir, exist_ok=True)

# Input files
teams_input = "../capturess_teams/teams_rtp_records_18_to_19.csv"
meet_input = "../capturess_teams/meet_rtp_records_18_to_19.csv"

# Process Teams
print(f"--- Starting Teams Processing ---")
teams_out = os.path.join(base_dir, "Teams")
split_csv(teams_input, "Teams", teams_out)

# Process Meet
print(f"--- Starting Meet Processing ---")
meet_out = os.path.join(base_dir, "Meet")
split_csv(meet_input, "Meet", meet_out)

print(f"All Done! Check the '{base_dir}' folder.")